<h1><b>IR ALEM 2: RECUPERAÇÃO DE CASOS COM EMBEDDINGS VISUAIS<b><h1>

In [ ]:
!pip install -q kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

<h2>Instalações e Imports<h2>

In [ ]:
import os
import gc
from typing import Tuple, List, Dict, Any

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as transforms

# Instalação correta e limpa do FAISS
!pip install -q faiss-cpu
import faiss

<h2>Configuração de Dataset, Transformações e Modelo Médico<h2>

In [ ]:
class ChestXRayDataset(Dataset):
    """Dataset otimizado para carregamento de Radiografias de Tórax."""
    def __init__(self, root_dir: str, transform: transforms.Compose = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths: List[str] = []
        self.labels: List[int] = []
        self.class_to_idx: Dict[str, int] = {"NORMAL": 0, "PNEUMONIA": 1}

        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"Diretório não encontrado: {root_dir}")

        self._load_dataset()

    def _load_dataset(self) -> None:
        for class_name, class_idx in self.class_to_idx.items():
            class_path = os.path.join(self.root_dir, class_name)
            if os.path.isdir(class_path):
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith(('.jpeg', '.png', '.jpg')):
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(class_idx)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int, str]:
        img_path = self.image_paths[idx]
        try:
            with Image.open(img_path) as img:
                image = img.convert('RGB')
        except Exception as e:
            # Fallback para evitar quebra em imagens corrompidas no dataset
            print(f"Erro ao carregar {img_path}: {e}")
            image = Image.new('RGB', (224, 224), color=0)

        if self.transform:
            image = self.transform(image)

        return image, self.labels[idx], img_path


def get_medical_feature_extractor() -> nn.Module:
    """
    Retorna uma ResNet-50 adaptada para imagens médicas.
    Modificação: Congelamos as primeiras camadas e focamos na extração de features texturais.
    """
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    # Remove a camada totalmente conectada (FC) mantendo o Pooling adaptativo
    feature_extractor = nn.Sequential(*(list(model.children())[:-1]))
    return feature_extractor

<h2>Pipeline de Extração de Embeddings (Com Gerenciamento de Memória)<h2>

In [ ]:
def extract_embeddings(dataloader: DataLoader, model: nn.Module, device: torch.device) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Extrai embeddings controlando o consumo de VRAM da GPU."""
    model.to(device)
    model.eval()

    raw_embeddings: List[np.ndarray] = []
    labels_list: List[int] = []
    paths_list: List[str] = []

    print("\n[INFO] Iniciando extração de embeddings visuais...")
    with torch.no_grad():
        for imgs, lbls, paths in tqdm(dataloader, desc="Processando Batches"):
            imgs = imgs.to(device, non_blocking=True)
            features = model(imgs)
            # Flatten eficiente
            features = torch.flatten(features, start_dim=1)

            raw_embeddings.append(features.cpu().numpy())
            labels_list.extend(lbls.numpy())
            paths_list.extend(paths)

    # Concatenação e tipagem estrita para o FAISS (float32)
    embeddings = np.vstack(raw_embeddings).astype('float32')
    labels = np.array(labels_list, dtype=np.int32)

    # Limpeza de cache da GPU
    del imgs, features
    torch.cuda.empty_cache()
    gc.collect()

    return embeddings, labels, paths_list

<h2>Indexação Avançada com FAISS<h2>

In [ ]:
class VectorIndexManager:
    """Gerenciador de índice vetorial utilizando FAISS com normalização interna."""
    def __init__(self, dimension: int):
        # IndexFlatIP + Normalização L2 = Similaridade de Cosseno (Melhor para embeddings)
        self.index = faiss.IndexFlatIP(dimension)

    def add_vectors(self, embeddings: np.ndarray) -> None:
        # Clonamos para não alterar a matriz original por referência
        vectors = embeddings.copy()
        faiss.normalize_L2(vectors)
        self.index.add(vectors)
        print(f"[INFO] {self.index.ntotal} vetores normalizados e indexados.")

    def search(self, query_embedding: np.ndarray, k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        q_vector = query_embedding.copy().reshape(1, -1)
        faiss.normalize_L2(q_vector)
        distances, indices = self.index.search(q_vector, k + 1)
        # Retorna pulando o primeiro elemento (que seria a própria imagem se ela estiver no índice)
        return distances[0][1:], indices[0][1:]

<h2>Execução do Experimento e Avaliação (Precision@K)K<h2>

In [ ]:
# 1. Configurações Globais
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_DIR = 'chest_xray/test'  # caminho do Kaggle
BATCH_SIZE = 64                  # Aumentado para maior paralelismo

# 2. Pipeline de Transformação Homologado
medical_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Inicialização dos Componentes
xray_dataset = ChestXRayDataset(root_dir=DATASET_DIR, transform=medical_transform)
xray_dataloader = DataLoader(xray_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

extractor = get_medical_feature_extractor()

# 4. Extração e Indexação
embeddings, labels, image_paths = extract_embeddings(xray_dataloader, extractor, DEVICE)

index_manager = VectorIndexManager(dimension=embeddings.shape[1])
index_manager.add_vectors(embeddings)

<h2>Visualização Analítica dos Resultados<h2>

In [ ]:
def plot_comparative_cbir_results(k: int = 5) -> None:
    """
    Localiza dinamicamente o primeiro caso de cada classe (Normal e Pneumonia)
    e plota um painel comparativo de duas linhas com seus respectivos 5 vizinhos.
    """
    # 1. Encontrar o primeiro índice de cada classe no dataset
    idx_normal = int(np.where(labels == 0)[0][0])
    idx_pneumonia = int(np.where(labels == 1)[0][0])
    queries = [idx_normal, idx_pneumonia]

    # Configuração da figura (2 linhas: uma para Normal, uma para Pneumonia)
    fig, axes = plt.subplots(2, k + 1, figsize=(22, 10))

    for row, query_idx in enumerate(queries):
        # Executar a busca no FAISS para a query atual
        distances, indices = index_manager.search(embeddings[query_idx], k=k)
        query_label = labels[query_idx]
        retrieved_labels = labels[indices]

        # Calcular Precision@K para esta linha
        correct_matches = np.sum(retrieved_labels == query_label)
        precision_at_k = (correct_matches / k) * 100

        class_title = "NORMAL" if query_label == 0 else "PNEUMONIA"

        # --- Plot da Imagem de Consulta (Coluna 0) ---
        with Image.open(image_paths[query_idx]) as q_img:
            axes[row, 0].imshow(q_img, cmap='gray')
        axes[row, 0].set_title(
            f"QUERY CASO {row+1}: RX {class_title}\nPrecision@{k}: {precision_at_k:.1f}%",
            color='blue', fontweight='bold', fontsize=11, pad=10
        )
        axes[row, 0].axis('off')

        # --- Plot dos 5 Vizinhos Recuperados (Colunas 1 a K) ---
        for i, idx in enumerate(indices):
            with Image.open(image_paths[idx]) as n_img:
                axes[row, i + 1].imshow(n_img, cmap='gray')

            is_correct = retrieved_labels[i] == query_label
            color = "green" if is_correct else "red"
            status = "CORRETO" if is_correct else "INCORRETO"
            neighbor_class = 'PNEUMONIA' if retrieved_labels[i] == 1 else 'NORMAL'

            axes[row, i + 1].set_title(
                f"Vizinho {i+1} ({status})\nClasse: {neighbor_class}\nSim: {distances[i]:.4f}",
                color=color, fontsize=9, pad=8
            )
            axes[row, i + 1].axis('off')

    # Ajustes finos de layout
    plt.subplots_adjust(wspace=0.15, hspace=0.3)
    plt.suptitle(
        "PAINEL COMPARATIVO DE RECUPERAÇÃO SEMÂNTICA (CBIR)\nAnálise Comparativa de Vizinhança para Casos Clínicos Diferenciados",
        fontsize=16, fontweight='bold', y=0.98
    )

    # Salva a imagem automaticamente em alta resolução
    plt.savefig('resultado_cbir_duplo.png', bbox_inches='tight', dpi=300)
    plt.show()

# gera o gráfico duplo
plot_comparative_cbir_results(k=5)
